# Study 11 - B2: paired bootstrap intervals (CRITICAL)

Section 2.6.2 states that differences are reported with paired bootstrap intervals on
common resamples. Tables 1 to 4 report bare point estimates. The headline positive result
is +0.0095 AUROC, and nothing currently establishes it differs from zero.

`study3` has `boot_auc()`, which bootstraps each model **separately**. Two separate
intervals cannot answer "is A better than B" because they discard the correlation between
the two scores on the same rows. This notebook resamples rows **once** and evaluates both
models on that same resample.

2000 resamples, not 300. A difference of 0.01 needs the resolution.


In [1]:
import numpy as np, pandas as pd, math, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
np.random.seed(0)
OUT=Path("../outputs/study11_bootstrap"); OUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({"figure.dpi":130,"savefig.dpi":150,"font.size":11,"axes.grid":True,"grid.alpha":0.25,"axes.spines.top":False,"axes.spines.right":False})
STEP=6; H=4; TAU=15.0
pm=pd.read_csv("../data/openaq_regional/regional_pm25.csv"); pm["datetime"]=pd.to_datetime(pm["datetime"],utc=True)
pm["name"]=pm.station.str.split("::").str[1]; pm=pm.dropna(subset=["value"]); pm=pm[(pm.value>=0)&(pm.value<1500)]
met=pd.read_csv("../data/era5/era5_regional.csv",usecols=["datetime","station","wind_speed","blh","t2m"])
met["datetime"]=pd.to_datetime(met["datetime"],utc=True); met=met.rename(columns={"station":"name"})
yrs=pm.groupby("station")["datetime"].agg(lambda s:(s.max()-s.min()).days/365.25); pm=pm[pm.station.isin(yrs[yrs>=2].index)].copy()
print("stations",pm.station.nunique())

stations 21


In [2]:
# ---- twin, metrics, helpers: copied verbatim from study4/study5 so behaviour is identical ----
K=5; LEV=[i/(K-1) for i in range(K)]; SDL=[1.0+0.5*l for l in LEV]; LOGC=0.5*math.log(2*math.pi); LGC=[-LOGC-math.log(sd) for sd in SDL]
def lgv(z): return [LGC[k]-0.5*((z-2.0*LEV[k])/SDL[k])**2 for k in range(K)]
WPM,WW,WB,WT=1.0,0.5,0.5,0.4
def twin(zpm,zw,zb,zt,wpm=None,ww=None,wb=None,wt=None):
    wpm=WPM if wpm is None else wpm; ww=WW if ww is None else ww
    wb=WB if wb is None else wb;   wt=WT if wt is None else wt
    logb=[-3*k for k in range(K)]; m=max(logb); ss=sum(math.exp(x-m) for x in logb); logb=[x-m-math.log(ss) for x in logb]
    risk=np.empty(len(zpm)); ent=np.empty(len(zpm))
    for t in range(len(zpm)):
        mb=max(logb); b=[math.exp(x-mb) for x in logb]; sb=sum(b); b=[x/sb for x in b]
        el=sum(b[k]*LEV[k] for k in range(K)); up=0.25; pdn=0.05 if el>0.4 else 0.15; bp=[0.0]*K
        for k in range(K):
            u=up if k<K-1 else 0.0; d=pdn if k>0 else 0.0; bp[k]+=b[k]*(1-u-d)
            if k<K-1: bp[k+1]+=b[k]*u
            if k>0: bp[k-1]+=b[k]*d
        logbp=[math.log(x+1e-12) for x in bp]
        for zv,w,inv in [(zpm,wpm,1),(zw,ww,-1),(zb,wb,-1),(zt,wt,-1)]:
            if not math.isnan(zv[t]):
                lv=lgv(inv*zv[t]); logbp=[logbp[k]+w*lv[k] for k in range(K)]
        m2=max(logbp); s2=math.log(sum(math.exp(x-m2) for x in logbp)); logb=[x-m2-s2 for x in logbp]
        eb=[math.exp(x) for x in logb]; risk[t]=sum(eb[k]*LEV[k] for k in range(K)); ent[t]=-sum(p*math.log(p+1e-12) for p in eb)
    return risk,ent
def auc(s,y):
    s=np.asarray(s,float);y=np.asarray(y,int);pos,neg=(y==1),(y==0)
    if pos.sum()==0 or neg.sum()==0:return np.nan
    o=np.argsort(s);r=np.empty(len(s));r[o]=np.arange(1,len(s)+1)
    u,inv,c=np.unique(s,return_inverse=True,return_counts=True);sm=np.zeros(len(c));np.add.at(sm,inv,r);r=(sm/c)[inv]
    return float((r[pos].sum()-pos.sum()*(pos.sum()+1)/2)/(pos.sum()*neg.sum()))
def apr(s,y):
    s=np.asarray(s,float);y=np.asarray(y,int);o=np.argsort(-s);y=y[o];tp=np.cumsum(y);fp=np.cumsum(1-y);prec=tp/(tp+fp);rec=tp/max(y.sum(),1)
    return float(np.sum((rec-np.r_[0,rec[:-1]])*prec))
def brier(p,y): return float(np.mean((np.asarray(p,float)-np.asarray(y,float))**2))
def sig(z): return 1/(1+np.exp(-np.clip(z,-30,30)))
def platt_fit(x,y,ep=3000,lr=.3):
    x=np.asarray(x,float);mu,sd=x.mean(),x.std()+1e-9;xs=(x-mu)/sd;w=0.;bi=0.
    for _ in range(ep): p=1/(1+np.exp(-(w*xs+bi)));w-=lr*np.mean((p-y)*xs);bi-=lr*np.mean(p-y)
    return lambda z:1/(1+np.exp(-(w*((np.asarray(z,float)-mu)/sd)+bi)))

In [3]:
FEATS=["zpm","zpm_l1","zpm_l2","zpm_tr","zwind","zblh","ztemp","hour_sin","hour_cos","doy_sin","doy_cos"]
ST={}; rows=[]
for st,g in pm.groupby("station"):
    name=g["name"].iloc[0]
    s=g.set_index("datetime").value.resample(f"{STEP}h").mean()
    mg=met[met.name==name].set_index("datetime")[["wind_speed","blh","t2m"]].resample(f"{STEP}h").mean()
    idx=s.index; A=pd.DataFrame({"pm":s}).join(mg).reindex(idx); tm=(idx.month%2==0)
    def z(col,log=False):
        v=np.log1p(np.clip(A[col].to_numpy(),0,None)) if log else A[col].to_numpy()
        mu,sd=np.nanmean(v[tm]),np.nanstd(v[tm])+1e-9; return (v-mu)/sd
    zpm=z("pm",log=True); zw=z("wind_speed"); zb=z("blh"); zt=z("t2m"); risk,ent=twin(zpm,zw,zb,zt)
    v=A["pm"].to_numpy(); hr=idx.hour.to_numpy(); doy=idx.dayofyear.to_numpy()
    ST[st]=dict(zpm=zpm,zw=zw,zb=zb,zt=zt,v=v,tm=tm,hr=hr,doy=doy,idx=idx)
    g2=lambda a:(a if not np.isnan(a) else 0.0)
    for t in range(2,len(v)-H):
        if np.isnan(v[t]) or v[t]>=TAU: continue
        fut=v[t+1:t+1+H]
        if np.all(np.isnan(fut)): continue
        rows.append(dict(station=st,ti=t,split=("train" if tm[t] else "test"),label=int(np.nanmax(fut)>=TAU),
            zpm=g2(zpm[t]),zpm_l1=g2(zpm[t-1]),zpm_l2=g2(zpm[t-2]),zpm_tr=g2(zpm[t]-zpm[t-2]),zwind=g2(zw[t]),zblh=g2(zb[t]),ztemp=g2(zt[t]),
            hour_sin=math.sin(2*math.pi*hr[t]/24),hour_cos=math.cos(2*math.pi*hr[t]/24),doy_sin=math.sin(2*math.pi*doy[t]/365),doy_cos=math.cos(2*math.pi*doy[t]/365),
            risk=risk[t],ent=ent[t]))
S=pd.DataFrame(rows); tr,te=S[S.split=="train"],S[S.split=="test"]; ytr,yte=tr.label.values,te.label.values
print("samples",len(S),"| test",len(te),"| prevalence %.3f"%S.label.mean())

samples 92888 | test 48208 | prevalence 0.073


## The paired bootstrap

In [4]:
# ---- paired bootstrap on COMMON resamples: the interval on the DIFFERENCE ----
# study3 has boot_auc(), which bootstraps each model separately. Two separate intervals
# cannot answer "is A better than B" because they ignore the correlation between the two
# scores on the same rows. This resamples rows once and evaluates both models on that
# same resample, which is what the manuscript claims to report.
def paired_boot(sa,sb,y,n=2000,seed=1):
    sa=np.asarray(sa,float); sb=np.asarray(sb,float); y=np.asarray(y,int)
    rng=np.random.default_rng(seed); d=[]
    for _ in range(n):
        i=rng.integers(0,len(y),len(y))
        yi=y[i]
        if not (0<yi.sum()<len(yi)): continue
        d.append(auc(sa[i],yi)-auc(sb[i],yi))
    d=np.array(d)
    lo,hi=np.percentile(d,[2.5,97.5])
    return dict(diff=float(auc(sa,y)-auc(sb,y)), boot_mean=float(d.mean()),
                lo95=float(lo), hi95=float(hi),
                excludes_zero=bool((lo>0) or (hi<0)), n_boot=int(len(d)))

def report(name,sa,sb,y,n=2000):
    r=paired_boot(sa,sb,y,n=n); r["comparison"]=name
    flag="SURVIVES" if r["excludes_zero"] else "PARITY (interval spans zero)"
    print(f'{name:52s} {r["diff"]:+.4f}  [{r["lo95"]:+.4f}, {r["hi95"]:+.4f}]  {flag}')
    return r

## Table 1 comparisons

In [5]:
# ---- Table 1: discrimination on complete data ----
gbm=HistGradientBoostingClassifier(max_iter=300,learning_rate=0.05,max_depth=4,l2_regularization=1.0,random_state=0).fit(tr[FEATS].values,ytr)
p_gbm=gbm.predict_proba(te[FEATS].values)[:,1]
ftw=platt_fit(tr.risk.values,ytr); p_twin=ftw(te.risk.values)
p_cur=te.zpm.values
print("comparison                                            diff        95% CI                verdict")
print("-"*104)
res=[report("monitor - GBM (Table 1, complete data)",p_twin,p_gbm,yte),
     report("monitor - current level (Table 1)",p_twin,p_cur,yte),
     report("GBM - current level (Table 1)",p_gbm,p_cur,yte)]
pd.DataFrame(res).to_csv(OUT/"b2_table1_intervals.csv",index=False)

comparison                                            diff        95% CI                verdict
--------------------------------------------------------------------------------------------------------
monitor - GBM (Table 1, complete data)               -0.0217  [-0.0273, -0.0162]  SURVIVES
monitor - current level (Table 1)                    +0.0348  [+0.0263, +0.0424]  SURVIVES
GBM - current level (Table 1)                        +0.0564  [+0.0498, +0.0632]  SURVIVES
